In [3]:
!pip install huggingface_hub -q

In [4]:
from huggingface_hub import login
login()  # Paste your HF token when prompted

In [10]:
!pip install transformers accelerate peft bitsandbytes -q

import os
import torch
import gc
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ✅ Base Phi-2 model
base_model_name = "microsoft/phi-2"

# ✅ Paths for LoRA adapters
aug_lora_path = "/kaggle/input/phi2/tensorflow2/default/1/aug_lora - Copie"
finetuned_lora_path = "/kaggle/input/phi2/tensorflow2/default/1/lora_finetuned - Copie"
hyperpar_root_path = "/kaggle/input/phi2/tensorflow2/default/1/lora_hyperpar - Copie"  # contains 3 subfolders

# ✅ Load dataset
dataset_path = "/kaggle/input/synthetic-dataset-v2/synthetic_dataset_v1.json"  # adjust path if needed
with open(dataset_path, "r") as f:
    data = json.load(f)

eval_data = data[:50]  # subset for quick evaluation

results = {}

# ✅ List all hyperparameter experiment subfolders
hyperparam_subfolders = [os.path.join(hyperpar_root_path, d) for d in os.listdir(hyperpar_root_path) if os.path.isdir(os.path.join(hyperpar_root_path, d))]
print(f"Found {len(hyperparam_subfolders)} hyperparameter variations: {hyperparam_subfolders}")

# ✅ Model configs (dynamic for hyperparam variations)
model_configs = {
    "zero_shot": {"type": "full", "path": None},
    "aug_lora": {"type": "lora", "path": aug_lora_path},
    "lora_finetuned": {"type": "lora", "path": finetuned_lora_path}
}

# Add each hyperparameter variation
for i, folder in enumerate(hyperparam_subfolders, start=1):
    model_configs[f"hyperpar_exp{i}"] = {"type": "lora", "path": folder}

# ✅ Evaluation Loop
for model_name, cfg in model_configs.items():
    print(f"\n🔍 Evaluating: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    if cfg["type"] == "full":
        # ✅ Zero-shot: load full Phi-2
        model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            load_in_4bit=True
        )
    else:
        # ✅ Load base Phi-2 and attach LoRA
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            load_in_4bit=True
        )
        model = PeftModel.from_pretrained(base_model, cfg["path"])

    predictions = []
    for idx, sample in enumerate(eval_data):
        prompt = f"Generate 3 domain names for this business. Business: {sample['business_description']}\nDomains:"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=50,
                temperature=0.7,
                top_p=0.95,
                do_sample=True
            )

        pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predictions.append({
            "index": idx,
            "business_description": sample['business_description'],
            "predicted": pred_text
        })

    results[model_name] = predictions
    print(f"✅ Done: {model_name}")

    # ✅ Free GPU memory
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 GPU memory cleared.")

# ✅ Save all predictions
with open("model_predictions.json", "w") as f:
    json.dump(results, f, indent=4)

print("\n✅ All models evaluated. Predictions saved to model_predictions.json.")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Found 3 hyperparameter variations: ['/kaggle/input/phi2/tensorflow2/default/1/lora_hyperpar - Copie/lr5e-4, r32, a64, d0.1', '/kaggle/input/phi2/tensorflow2/default/1/lora_hyperpar - Copie/lr2e-4, r16, a32, d0.05', '/kaggle/input/phi2/tensorflow2/default/1/lora_hyperpar - Copie/lr1e-4, r8, a16, d0.05']

🔍 Evaluating: zero_shot


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

...

✅ Done: hyperpar_exp2
🧹 GPU memory cleared.

🔍 Evaluating: hyperpar_exp3


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

✅ Done: hyperpar_exp3
🧹 GPU memory cleared.

✅ All models evaluated. Predictions saved to model_predictions.json.


In [12]:
from IPython.display import FileLinks

# Create a download link for the entire working directory
FileLinks('/kaggle/working')

/kaggle/working/
  model_predictions.json